# Générer les Offsets UNE FOIS
Exécuter sur Colab, télécharger le fichier, puis tout en local

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')

In [ ]:
!pip install transformers -q

In [ ]:
import json
from transformers import AutoTokenizer

# Load corpus
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Load boundary tokens
with open('results/camelbert_boundary_tokens_clean.json', 'r', encoding='utf-8') as f:
    boundary_data = json.load(f)

boundary_indices = set(boundary_data['boundary_indices'])

# Tokenize with offsets
print("Tokenizing...")
tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/bert-base-arabic-camelbert-msa')
encoded = tokenizer(
    text,
    return_tensors='pt',
    return_offsets_mapping=True,
    truncation=False,
    padding=False,
)

token_ids = encoded['input_ids'][0].numpy()
offsets = encoded['offset_mapping'][0].numpy()

print(f"Total tokens: {len(token_ids):,}")
print(f"Boundary tokens: {len(boundary_indices):,}")

In [ ]:
# Create offsets mapping
print("Creating offsets mapping...")

offsets_list = []
for idx in boundary_indices:
    if idx < len(offsets):
        char_start, char_end = offsets[idx]
        offsets_list.append({
            'token_index': idx,
            'char_start': int(char_start),
            'char_end': int(char_end),
            'token_text': tokenizer.decode([token_ids[idx]])
        })

print(f"Offsets created: {len(offsets_list)}")

In [ ]:
# Save ONLY offsets (lightweight)
results = {
    'corpus': 'kitab_uqala_reference_corpus.txt',
    'total_tokens': len(token_ids),
    'boundary_count': len(offsets_list),
    'offsets': offsets_list,  # ONLY what we need
}

with open('results/camelbert_boundary_offsets.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✓ Saved: results/camelbert_boundary_offsets.json")
print(f"\nFile size: {len(json.dumps(results))/1024:.1f} KB")

In [ ]:
from google.colab import files
files.download('results/camelbert_boundary_offsets.json')
print("Downloaded!")